In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# File paths
metric_fp = "metric.csv"
trace_fp = "trace.csv"
log_fp = "log.csv"
error_fp = "error_logs.csv"

# Load CSVs
df_metric = pd.read_csv(metric_fp)
df_trace = pd.read_csv(trace_fp)
df_log = pd.read_csv(log_fp)
df_err = pd.read_csv(error_fp)

# Parse timestamps to UTC datetimes
for df in (df_metric, df_trace, df_log, df_err):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to normalize pandas/numpy scalars to native Python types for JSON-serializable output
def normalize_records(records):
    norm = []
    for rec in records:
        newrec = {}
        for k, v in rec.items():
            if pd.isna(v):
                newrec[k] = None
            elif isinstance(v, (np.generic,)):
                newrec[k] = v.item()
            else:
                newrec[k] = v
        norm.append(newrec)
    return norm

# Summarize function for metric/trace/log (group by cmdb_id + kpi column)
def summarize_kpi_file(df, kpi_col, top_n=20, include_percentiles=False):
    summary = {}
    if df.empty:
        summary['min_timestamp'] = None
        summary['max_timestamp'] = None
        summary['unique_cmdb_id'] = []
        summary['top_pairs'] = []
        return summary

    summary['min_timestamp'] = df['timestamp'].min().isoformat()
    summary['max_timestamp'] = df['timestamp'].max().isoformat()
    summary['unique_cmdb_id'] = sorted(df['cmdb_id'].dropna().unique().tolist())

    grp = df.groupby(['cmdb_id', kpi_col])
    if include_percentiles:
        cnt = grp['value'].count()
        p95 = grp['value'].quantile(0.95)
        p5 = grp['value'].quantile(0.05)
        agg = pd.concat([cnt.rename('count'), p95.rename('p95'), p5.rename('p5')], axis=1).reset_index()
        agg = agg.sort_values('count', ascending=False).head(top_n)
        # Ensure native types
        records = agg.to_dict(orient='records')
        records = normalize_records(records)
        summary['top_pairs'] = records
    else:
        # only count
        agg = grp.size().reset_index(name='count').sort_values('count', ascending=False).head(top_n)
        records = agg.to_dict(orient='records')
        records = normalize_records(records)
        summary['top_pairs'] = records

    return summary

# Build summaries
metric_summary = summarize_kpi_file(df_metric, 'kpi_name', top_n=20, include_percentiles=True)
trace_summary = summarize_kpi_file(df_trace, 'trace_name', top_n=20, include_percentiles=True)
log_summary = summarize_kpi_file(df_log, 'log_name', top_n=20, include_percentiles=False)

# error_logs: top 20 rows sorted by timestamp (ISO UTC strings)
err_summary = {}
if df_err.empty:
    err_summary['min_timestamp'] = None
    err_summary['max_timestamp'] = None
    err_summary['unique_cmdb_id'] = []
    err_summary['top_rows'] = []
else:
    err_summary['min_timestamp'] = df_err['timestamp'].min().isoformat()
    err_summary['max_timestamp'] = df_err['timestamp'].max().isoformat()
    err_summary['unique_cmdb_id'] = sorted(df_err['cmdb_id'].dropna().unique().tolist())
    top_rows = df_err.sort_values('timestamp').head(20).copy()
    top_rows['timestamp'] = top_rows['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    # select requested columns
    top_rows = top_rows[['timestamp', 'cmdb_id', 'message']]
    records = top_rows.to_dict(orient='records')
    records = normalize_records(records)
    err_summary['top_rows'] = records

# Compose final JSON-serializable dict
results_summary = {
    'metric.csv': metric_summary,
    'trace.csv': trace_summary,
    'log.csv': log_summary,
    'error_logs.csv': err_summary
}

results_summary

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry (all timestamps UTC):\n\n"
    "metric.csv: Data covers 2024-01-18T18:24:00+00:00 to 2024-01-18T18:48:00+00:00. "
    "Observed components: ['adservice', 'cartservice', 'checkoutservice', 'currencyservice', 'emailservice', "
    "'frontend', 'frontend-external', 'paymentservice', 'productcatalogservice', 'recommendationservice', 'redis', 'shippingservice']. "
    "Top metric pairs (by sample count, p95 / p5 shown where available):\n"
    "- adservice: cpu (count=25, p95≈0.778, p5≈0.637), diskio (p95≈69795, p5≈45909), mem (p95≈110,516,074, p5≈110,187,684), socket (4.0), workload (p95≈15.78)\n"
    "- cartservice: cpu (count=25, p95≈2.939, p5≈1.979), mem (p95≈43,632,080), socket (5.0), workload (p95≈26.53)\n"
    "- checkoutservice: cpu (count=25, p95≈0.510, p5≈0.309), latency-50/90 present (p95 up to ≈0.27), mem (p95≈12,447,034), socket (9.0)\n\n"
    "trace.csv: Data covers 2024-01-18T18:24:00+00:00 to 2024-01-18T18:48:00+00:00. "
    "Observed components: ['checkoutservice', 'currencyservice', 'emailservice', 'frontendservice', 'paymentservice', 'productcatalogservice', 'recommendationservice', 'root']. "
    "Top trace pairs are dominated by checkoutservice traces (each count=25). Example values:\n"
    "- checkoutservice.trace.from_checkoutservice.duration_mean (count=25, p95≈0.0118, p5≈0.0039)\n"
    "- checkoutservice.trace.from_frontendservice.duration_mean (p95≈0.1586, p5≈0.0796)\n"
    "- several checkoutservice trace metrics have low latencies (p95 typically <0.3s) and many zero error_rate p95/p5 values.\n\n"
    "log.csv: Data covers 2024-01-18T18:24:00+00:00 to 2024-01-18T18:48:00+00:00. "
    "Observed components include adservice, cartservice, checkoutservice, currencyservice, emailservice, frontend, paymentservice, recommendationservice, redis, shippingservice. "
    "Top log pairs (each typically count=25) are log.error_count and log.row_count for most services. Redis has only 5 samples each.\n\n"
    "error_logs.csv: No rows returned (empty top_rows); no timestamps or components present.\n\n"
    "Overall quick takeaway: metric data shows high sampling for adservice, cartservice, and checkoutservice with measurable CPU/memory/diskio/workload values; trace data is concentrated on checkoutservice with low latencies; logs show per-service error_count and row_count consistently. error_logs has no entries."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'min_timestamp': '2024-01-18T18:24:00+00:00', 'max_timestamp': '2024-01-18T18:48:00+00:00', 'unique_cmdb_id': ['adservice', 'cartservice', 'checkoutservice', 'currencyservice', 'emailservice', 'frontend', 'frontend-external', 'paymentservice', 'productcatalogservice', 'recommendationservice', 'redis', 'shippingservice'], 'top_pairs': [{'cmdb_id': 'adservice', 'kpi_name': 'cpu', 'count': 25, 'p95': 0.7781403863157202, 'p5': 0.6370397478487215}, {'cmdb_id': 'adservice', 'kpi_name': 'diskio', 'count': 25, 'p95': 69795.44423620752, 'p5': 45909.19134925312}, {'cmdb_id': 'adservice', 'kpi_name': 'latency-50', 'count': 25, 'p95': 0.00300704064640524, 'p5': 0.00298197022201926}, {'cmdb_id': 'adservice', 'kpi_name': 'latency-90', 'count': 25, 'p95': 0.00462482299400596, 'p5': 0.0046015178203158}, {'cmdb_id': 'adservice', 'kpi_name': 'mem', 'count': 25, 'p95': 110516073.81333335, 'p5': 110187683.84}, {'cmdb_id': 'adservice', 'kpi_name': 'socket', 'count': 25, 'p95': 4.0, 'p5': 4.0}, {'cmdb_id': 'adservice', 'kpi_name': 'workload', 'count': 25, 'p95': 15.782566666666668, 'p5': 14.611833333333333}, {'cmdb_id': 'cartservice', 'kpi_name': 'cpu', 'count': 25, 'p95': 2.938951536509825, 'p5': 1.9789676639612965}, {'cmdb_id': 'cartservice', 'kpi_name': 'latency-50', 'count': 25, 'p95': 0.00313590321814414, 'p5': 0.0030604273547385798}, {'cmdb_id': 'cartservice', 'kpi_name': 'latency-90', 'count': 25, 'p95': 0.004844625792659619, 'p5': 0.0047087692385295}, {'cmdb_id': 'cartservice', 'kpi_name': 'mem', 'count': 25, 'p95': 43632080.21333333, 'p5': 43411374.08}, {'cmdb_id': 'cartservice', 'kpi_name': 'socket', 'count': 25, 'p95': 5.0, 'p5': 5.0}, {'cmdb_id': 'cartservice', 'kpi_name': 'workload', 'count': 25, 'p95': 26.52582, 'p5': 23.37953}, {'cmdb_id': 'checkoutservice', 'kpi_name': 'cpu', 'count': 25, 'p95': 0.5097389126499624, 'p5': 0.3087726103227389}, {'cmdb_id': 'checkoutservice', 'kpi_name': 'diskio', 'count': 25, 'p95': 0.0, 'p5': 0.0}, {'cmdb_id': 'checkoutservice', 'kpi_name': 'latency-50', 'count': 25, 'p95': 0.1598725826265947, 'p5': 0.07378650793650784}, {'cmdb_id': 'checkoutservice', 'kpi_name': 'latency-90', 'count': 25, 'p95': 0.26936210719998827, 'p5': 0.20127624916363304}, {'cmdb_id': 'checkoutservice', 'kpi_name': 'mem', 'count': 25, 'p95': 12447034.026666665, 'p5': 11889704.96}, {'cmdb_id': 'checkoutservice', 'kpi_name': 'socket', 'count': 25, 'p95': 9.0, 'p5': 9.0}, {'cmdb_id': 'checkoutservice', 'kpi_name': 'workload', 'count': 25, 'p95': 1.25312, 'p5': 0.5982900000000001}]}, 'trace.csv': {'min_timestamp': '2024-01-18T18:24:00+00:00', 'max_timestamp': '2024-01-18T18:48:00+00:00', 'unique_cmdb_id': ['checkoutservice', 'currencyservice', 'emailservice', 'frontendservice', 'paymentservice', 'productcatalogservice', 'recommendationservice', 'root'], 'top_pairs': [{'cmdb_id': 'checkoutservice', 'trace_name': 'trace.from_checkoutservice.duration_mean', 'count': 25, 'p95': 0.011795954545454533, 'p5': 0.00389291462091466}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.from_checkoutservice.duration_p95', 'count': 25, 'p95': 0.0722222999999999, 'p5': 0.0063228799999999205}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.from_checkoutservice.error_rate', 'count': 25, 'p95': 0.0, 'p5': 0.0}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.from_checkoutservice.row_count', 'count': 25, 'p95': 302.99999999999994, 'p5': 164.0}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.from_frontendservice.duration_mean', 'count': 25, 'p95': 0.15861656, 'p5': 0.07961602380952378}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.from_frontendservice.duration_p95', 'count': 25, 'p95': 0.2761994599999999, 'p5': 0.11939326}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.from_frontendservice.error_rate', 'count': 25, 'p95': 0.0, 'p5': 0.0}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.from_frontendservice.row_count', 'count': 25, 'p95': 36.599999999999994, 'p5': 19.2}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.from_root.duration_mean', 'count': 25, 'p95': 0.01583772930245264, 'p5': 0.0108012524078862}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.from_root.duration_p95', 'count': 25, 'p95': 0.0722037799999999, 'p5': 0.05802109999999992}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.from_root.error_rate', 'count': 25, 'p95': 0.0, 'p5': 0.0}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.from_root.row_count', 'count': 25, 'p95': 145.39999999999998, 'p5': 89.6}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.to_checkoutservice.duration_mean', 'count': 25, 'p95': 0.011795954545454533, 'p5': 0.00389291462091466}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.to_checkoutservice.duration_p95', 'count': 25, 'p95': 0.0722222999999999, 'p5': 0.0063228799999999205}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.to_checkoutservice.error_rate', 'count': 25, 'p95': 0.0, 'p5': 0.0}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.to_checkoutservice.row_count', 'count': 25, 'p95': 302.99999999999994, 'p5': 164.0}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.to_currencyservice.duration_mean', 'count': 25, 'p95': 0.0018295332057415596, 'p5': 0.00013281360079666}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.to_currencyservice.duration_p95', 'count': 25, 'p95': 0.00029435999999999997, 'p5': 0.00018579999999998}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.to_currencyservice.error_rate', 'count': 25, 'p95': 0.0, 'p5': 0.0}, {'cmdb_id': 'checkoutservice', 'trace_name': 'trace.to_currencyservice.row_count', 'count': 25, 'p95': 121.39999999999999, 'p5': 66.4}]}, 'log.csv': {'min_timestamp': '2024-01-18T18:24:00+00:00', 'max_timestamp': '2024-01-18T18:48:00+00:00', 'unique_cmdb_id': ['adservice', 'cartservice', 'checkoutservice', 'currencyservice', 'emailservice', 'frontend', 'paymentservice', 'recommendationservice', 'redis', 'shippingservice'], 'top_pairs': [{'cmdb_id': 'adservice', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'adservice', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'cartservice', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'cartservice', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'checkoutservice', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'checkoutservice', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'currencyservice', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'currencyservice', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'emailservice', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'emailservice', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'frontend', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'frontend', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'paymentservice', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'paymentservice', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'recommendationservice', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'recommendationservice', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'shippingservice', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'shippingservice', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'redis', 'log_name': 'log.row_count', 'count': 5}, {'cmdb_id': 'redis', 'log_name': 'log.error_count', 'count': 5}]}, 'error_logs.csv': {'min_timestamp': None, 'max_timestamp': None, 'unique_cmdb_id': [], 'top_rows': []}}```
```

In [ ]:
# In[2]:

# Reuse previously loaded df_metric (timestamps already parsed to UTC)
import pandas as pd
import numpy as np

# Components of interest
comps = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]

# Ensure df_metric exists
# Filter global series to only these components (for computing global thresholds we must use entire metric.csv but only for these components)
df_metric_sub = df_metric[df_metric['cmdb_id'].isin(comps)].copy()

# 1) Compute global statistics per (cmdb_id, kpi_name): sample count, global P95 and P5
grp = df_metric_sub.groupby(['cmdb_id','kpi_name'])['value']
global_count = grp.count().rename('global_count')
global_p95 = grp.quantile(0.95).rename('p95')
global_p5 = grp.quantile(0.05).rename('p5')

global_stats = pd.concat([global_count, global_p95, global_p5], axis=1).reset_index()

# 2) Filter metric.csv to incident window (UTC)
start = pd.to_datetime("2024-01-18 18:21:04", utc=True)
end   = pd.to_datetime("2024-01-18 18:51:04", utc=True)
df_window = df_metric[(df_metric['timestamp'] >= start) & (df_metric['timestamp'] <= end) & (df_metric['cmdb_id'].isin(comps))].copy()

# Prepare window-level aggregates
window_agg = df_window.groupby(['cmdb_id','kpi_name']).agg(
    count_in_window = ('value','count'),
    max_in_window = ('value','max'),
    min_in_window = ('value','min')
).reset_index()

# Merge window rows with global stats to evaluate anomalies against global p95/p5
df_win_join = df_window.merge(global_stats, on=['cmdb_id','kpi_name'], how='left')

# Create anomaly flags based on global thresholds (calculated BEFORE filtering)
df_win_join['above'] = df_win_join['value'] > df_win_join['p95']
df_win_join['below'] = df_win_join['value'] < df_win_join['p5']

# Aggregate anomaly counts and earliest timestamps per pair
anom_counts = df_win_join.groupby(['cmdb_id','kpi_name']).agg(
    count_above = ('above','sum'),
    count_below = ('below','sum'),
    earliest_above_ts = ('timestamp', lambda s: s[df_win_join.loc[s.index,'above']].min() if s[df_win_join.loc[s.index,'above']].any() else pd.NaT),
    earliest_below_ts = ('timestamp', lambda s: s[df_win_join.loc[s.index,'below']].min() if s[df_win_join.loc[s.index,'below']].any() else pd.NaT)
).reset_index()

# Merge all: global_stats, window_agg, anom_counts
merged = global_stats.merge(window_agg, on=['cmdb_id','kpi_name'], how='left').merge(anom_counts, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for window metrics/counts with zeros or NaT
merged['count_in_window'] = merged['count_in_window'].fillna(0).astype(int)
merged['max_in_window'] = merged['max_in_window']  # keep NaN if no samples
merged['min_in_window'] = merged['min_in_window']
merged['count_above'] = merged['count_above'].fillna(0).astype(int)
merged['count_below'] = merged['count_below'].fillna(0).astype(int)

# Format earliest timestamps to ISO UTC strings or None
def fmt_ts(x):
    if pd.isna(x):
        return None
    # x is timezone-aware Timestamp
    return pd.to_datetime(x).isoformat()

merged['earliest_above'] = merged['earliest_above_ts'].apply(fmt_ts)
merged['earliest_below'] = merged['earliest_below_ts'].apply(fmt_ts)
merged.drop(columns=['earliest_above_ts','earliest_below_ts'], inplace=True)

# Compute total anomaly count and filter rows where any anomalies occurred
merged['total_anomalies'] = merged['count_above'] + merged['count_below']
result_df = merged[ (merged['count_above']>0) | (merged['count_below']>0) ].copy()

# Keep requested columns and include global stats
cols = ['cmdb_id','kpi_name','global_count','p95','p5',
        'count_in_window','count_above','earliest_above','count_below','earliest_below',
        'max_in_window','min_in_window','total_anomalies']
result_df = result_df[cols]

# Sort by total anomaly count descending and limit to top 50
result_df = result_df.sort_values('total_anomalies', ascending=False).head(50).reset_index(drop=True)

# Keep output compact: ensure numeric types are native (optional)
result_df['global_count'] = result_df['global_count'].astype(int)
result_df['count_in_window'] = result_df['count_in_window'].astype(int)
result_df['count_above'] = result_df['count_above'].astype(int)
result_df['count_below'] = result_df['count_below'].astype(int)
result_df['total_anomalies'] = result_df['total_anomalies'].astype(int)

# Final variable to display
result_df

```
Out[2]:
```
Summary (analysis restricted to checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice; timestamps UTC):

- Analysis window: 2024-01-18 18:21:04 → 2024-01-18 18:51:04 UTC.
- Global thresholds were computed on the full metric series (global P95 and P5) for each (cmdb_id, kpi_name) before window filtering.
- Rows returned: 26 (only (cmdb_id, kpi_name) pairs with anomalies where value > global P95 or value < global P5).

Typical pattern observed:
- Most flagged pairs have global_count = 25 (all samples) and count_in_window = 25.
- Many pairs show 2 samples above the global P95 and 2 samples below the global P5 (so total_anomalies = 4).
- Earliest anomaly timestamps vary by KPI; examples below use ISO UTC.

Representative examples:
- checkoutservice — cpu: global_count=25, p95≈0.5097, p5≈0.3088; in-window count=25; count_above=2 (earliest above 2024-01-18T18:24:00+00:00), count_below=2 (earliest below 2024-01-18T18:35:00+00:00); max≈0.5574, min≈0.2961; total_anomalies=4.
- checkoutservice — latency-50: global_count=25, p95≈0.1599, p5≈0.0738; in-window count=25; count_above=2 (earliest above 18:39), count_below=2 (earliest below 18:25); total_anomalies=4.
- checkoutservice — latency-90, mem, workload: similar pattern (each with 2 above / 2 below, total_anomalies=4).
- recommendationservice — cpu / mem / latency-90 / workload: similar pattern (usually 2 above & 2 below; total_anomalies=4).
- emailservice — diskio: global_count=25, p95≈1.0339e10, p5=0; in-window count=25; count_above=2 (earliest above 2024-01-18T18:43:00+00:00), count_below=0; total_anomalies=2.

Overall takeaway:
- Multiple KPIs across the five services show deviations relative to their global P95/P5, most commonly 2 samples above P95 and 2 below P5 within the incident window.
- Checkoutservice and recommendationservice show multiple KPI anomalies (latency, cpu, mem, workload). Emailservice’s diskio shows anomalies only above P95 in this window.
- If you want the full table or specific top offenders (all 26 rows), I can present the compact JSON/table of those rows.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  global_count           p95            p5  count_in_window  count_above             earliest_above  count_below             earliest_below  max_in_window  min_in_window  total_anomalies
0         checkoutservice         cpu            25  5.097389e-01  3.087726e-01               25            2  2024-01-18T18:24:00+00:00            2  2024-01-18T18:35:00+00:00   5.574124e-01   2.960825e-01                4
1         checkoutservice  latency-50            25  1.598726e-01  7.378651e-02               25            2  2024-01-18T18:39:00+00:00            2  2024-01-18T18:25:00+00:00   1.621988e-01   5.858135e-02                4
2         checkoutservice  latency-90            25  2.693621e-01  2.012762e-01               25            2  2024-01-18T18:41:00+00:00            2  2024-01-18T18:25:00+00:00   3.103126e-01   1.848999e-01                4
3         checkoutservice         mem            25  1.244703e+07  1.188970e+07               25            2  2024-01-18T18:47:00+00:00            2  2024-01-18T18:26:00+00:00   1.264435e+07   1.181792e+07                4
4         checkoutservice    workload            25  1.253120e+00  5.982900e-01               25            2  2024-01-18T18:31:00+00:00            2  2024-01-18T18:36:00+00:00   1.269917e+00   3.598000e-01                4
..                    ...         ...           ...           ...           ...              ...          ...                        ...          ...                        ...            ...            ...              ...
21  recommendationservice         cpu            25  2.979276e+00  2.638404e+00               25            2  2024-01-18T18:45:00+00:00            2  2024-01-18T18:34:00+00:00   3.067139e+00   2.583652e+00                4
22  recommendationservice         mem            25  4.549231e+07  4.538621e+07               25            2  2024-01-18T18:31:00+00:00            2  2024-01-18T18:40:00+00:00   4.549393e+07   4.536716e+07                4
23  recommendationservice  latency-90            25  9.661227e-03  9.542975e-03               25            2  2024-01-18T18:37:00+00:00            2  2024-01-18T18:30:00+00:00   9.677203e-03   9.526289e-03                4
24  recommendationservice    workload            25  1.941101e+01  1.754836e+01               25            2  2024-01-18T18:33:00+00:00            2  2024-01-18T18:25:00+00:00   2.002667e+01   1.726333e+01                4
25           emailservice      diskio            25  1.033889e+10  0.000000e+00               25            2  2024-01-18T18:43:00+00:00            0                       None   1.080594e+10   0.000000e+00                2

[26 rows x 13 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Services of interest
comps = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]

# Reuse df_trace (assumed loaded and timestamp-parsed in prior steps)
df_trace_sub = df_trace[df_trace['cmdb_id'].isin(comps)].copy()

# 1) Global statistics (computed on the full trace series for these components)
grp_full = df_trace_sub.groupby(['cmdb_id','trace_name'])['value']
global_count = grp_full.count().rename('global_count')
global_p95 = grp_full.quantile(0.95).rename('global_p95')
global_p5  = grp_full.quantile(0.05).rename('global_p5')

global_stats = pd.concat([global_count, global_p95, global_p5], axis=1).reset_index()

# 2) Filter to the incident window (UTC)
start = pd.to_datetime("2024-01-18 18:21:04", utc=True)
end   = pd.to_datetime("2024-01-18 18:51:04", utc=True)
df_win = df_trace_sub[(df_trace_sub['timestamp'] >= start) & (df_trace_sub['timestamp'] <= end)].copy()

# Merge window rows with global thresholds (thresholds computed before filtering)
df_win = df_win.merge(global_stats, on=['cmdb_id','trace_name'], how='left')

# Flag anomalies relative to global p95/p5
df_win['above'] = (df_win['value'] > df_win['global_p95']).astype(int)
df_win['below'] = (df_win['value'] < df_win['global_p5']).astype(int)

# Window-level aggregations
agg_win = df_win.groupby(['cmdb_id','trace_name']).agg(
    count_in_window = ('value','count'),
    count_above = ('above','sum'),
    count_below = ('below','sum'),
    max_in_window = ('value','max'),
    min_in_window = ('value','min')
).reset_index()

# Earliest timestamps for above / below anomalies (if any)
earliest_above = df_win[df_win['above']==1].groupby(['cmdb_id','trace_name'])['timestamp'].min().rename('earliest_above_ts')
earliest_below = df_win[df_win['below']==1].groupby(['cmdb_id','trace_name'])['timestamp'].min().rename('earliest_below_ts')

# Combine everything
result = global_stats.merge(agg_win, on=['cmdb_id','trace_name'], how='left')
result = result.merge(earliest_above.reset_index(), on=['cmdb_id','trace_name'], how='left')
result = result.merge(earliest_below.reset_index(), on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for counts (no samples in window -> count_in_window NaN -> set 0)
result['count_in_window'] = result['count_in_window'].fillna(0).astype(int)
result['count_above'] = result['count_above'].fillna(0).astype(int)
result['count_below'] = result['count_below'].fillna(0).astype(int)

# Format earliest timestamps to ISO UTC strings or None
def fmt_iso(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).isoformat()

result['earliest_above'] = result['earliest_above_ts'].apply(fmt_iso)
result['earliest_below'] = result['earliest_below_ts'].apply(fmt_iso)
result.drop(columns=['earliest_above_ts','earliest_below_ts'], inplace=True)

# Compute totals and filter rows with anomalies
result['total_anomalies'] = result['count_above'] + result['count_below']
out = result[result['total_anomalies']>0].copy()

# Select and rename columns for compact output
out = out[[
    'cmdb_id','trace_name','global_count','global_p95','global_p5',
    'count_in_window','count_above','earliest_above','count_below','earliest_below',
    'max_in_window','min_in_window','total_anomalies'
]]

# Sort by total_anomalies desc and limit to top 20
out = out.sort_values('total_anomalies', ascending=False).head(20).reset_index(drop=True)

# Ensure integer types for counts
out['global_count'] = out['global_count'].astype(int)
out['count_in_window'] = out['count_in_window'].astype(int)
out['count_above'] = out['count_above'].astype(int)
out['count_below'] = out['count_below'].astype(int)
out['total_anomalies'] = out['total_anomalies'].astype(int)

# Final compact DataFrame output
trace_anomalies = out
trace_anomalies

```
Out[3]:
```
Summary of trace analysis (UTC):

- Scope & method: global P95/P5 were computed over the full trace.csv for the requested services. Then trace rows inside the incident window 2024-01-18 18:21:04 → 2024-01-18 18:51:04 UTC were checked for values > global P95 or < global P5. Only trace pairs with anomalies are reported.

- Results snapshot: the top 20 anomaly rows (all returned rows shown) are dominated by checkoutservice traces (both "from_" and "to_" relationships). Each reported pair has global_count = 25 and count_in_window = 25 (full coverage in the window). Most pairs have:
  - count_above = 2 and count_below = 2 (so total_anomalies = 4).
  - earliest above and below timestamps in the 18:24–18:43 UTC range (examples below).
  - anomalies affect both duration metrics and row_count metrics.

- Representative examples:
  - checkoutservice · trace.from_checkoutservice.duration_mean — global_p95 ≈ 0.0118, global_p5 ≈ 0.00389; in-window: 2 samples > p95 (earliest above 2024-01-18T18:35:00+00:00), 2 samples < p5 (earliest below 2024-01-18T18:32:00+00:00); max≈0.01456, min≈0.00386; total_anomalies=4.
  - checkoutservice · trace.from_frontendservice.duration_mean — global_p95 ≈ 0.1586, p5 ≈ 0.0796; 2 above (earliest 18:35), 2 below (earliest 18:25); total_anomalies=4.
  - checkoutservice · trace.from_checkoutservice.row_count — global_p95 = 303, p5 = 164; in-window max=316, min=22; 2 above and 2 below; total_anomalies=4.
  - checkoutservice · trace.to_emailservice.duration_mean — global_p95 ≈ 0.01190, p5 ≈ 0.000292; shows both very small values and spikes (2 above / 2 below), total_anomalies=4.

- Overall takeaway: anomalies are concentrated in checkoutservice trace metrics (durations and row counts) during the incident window, typically showing two samples above the global P95 and two below the global P5 for many trace metrics. The pattern suggests unstable or variable request/processing behavior in checkoutservice during the window. Other requested services (currencyservice, emailservice, productcatalogservice, recommendationservice) did not surface rows with total_anomalies > 0 in the top-20 output (anomalies appear as checkoutservice → other-service traces). 

If you want, I can provide the full compact table of the top 20 rows (timestamps and values included) or expand to include any non-checkoutservice anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                trace_name  global_count  global_p95   global_p5  count_in_window  count_above             earliest_above  count_below             earliest_below  max_in_window  min_in_window  total_anomalies
0   checkoutservice  trace.from_checkoutservice.duration_mean            25    0.011796    0.003893               25            2  2024-01-18T18:35:00+00:00            2  2024-01-18T18:32:00+00:00       0.014555       0.003863                4
1   checkoutservice   trace.from_checkoutservice.duration_p95            25    0.072222    0.006323               25            2  2024-01-18T18:42:00+00:00            2  2024-01-18T18:24:00+00:00       0.076677       0.005920                4
2   checkoutservice      trace.from_checkoutservice.row_count            25  303.000000  164.000000               25            2  2024-01-18T18:33:00+00:00            2  2024-01-18T18:36:00+00:00     316.000000      22.000000                4
3   checkoutservice  trace.from_frontendservice.duration_mean            25    0.158617    0.079616               25            2  2024-01-18T18:35:00+00:00            2  2024-01-18T18:25:00+00:00       0.173953       0.074332                4
4   checkoutservice   trace.from_frontendservice.duration_p95            25    0.276199    0.119393               25            2  2024-01-18T18:40:00+00:00            2  2024-01-18T18:24:00+00:00       0.301508       0.108981                4
5   checkoutservice      trace.from_frontendservice.row_count            25   36.600000   19.200000               25            2  2024-01-18T18:33:00+00:00            2  2024-01-18T18:36:00+00:00      37.000000       3.000000                4
6   checkoutservice             trace.from_root.duration_mean            25    0.015838    0.010801               25            2  2024-01-18T18:27:00+00:00            2  2024-01-18T18:25:00+00:00       0.016078       0.010169                4
7   checkoutservice              trace.from_root.duration_p95            25    0.072204    0.058021               25            2  2024-01-18T18:40:00+00:00            2  2024-01-18T18:43:00+00:00       0.074370       0.042763                4
8   checkoutservice                 trace.from_root.row_count            25  145.400000   89.600000               25            2  2024-01-18T18:33:00+00:00            2  2024-01-18T18:36:00+00:00     155.000000       8.000000                4
9   checkoutservice    trace.to_checkoutservice.duration_mean            25    0.011796    0.003893               25            2  2024-01-18T18:35:00+00:00            2  2024-01-18T18:32:00+00:00       0.014555       0.003863                4
10  checkoutservice     trace.to_checkoutservice.duration_p95            25    0.072222    0.006323               25            2  2024-01-18T18:42:00+00:00            2  2024-01-18T18:24:00+00:00       0.076677       0.005920                4
11  checkoutservice        trace.to_checkoutservice.row_count            25  303.000000  164.000000               25            2  2024-01-18T18:33:00+00:00            2  2024-01-18T18:36:00+00:00     316.000000      22.000000                4
12  checkoutservice    trace.to_currencyservice.duration_mean            25    0.001830    0.000133               25            2  2024-01-18T18:32:00+00:00            2  2024-01-18T18:40:00+00:00       0.002267       0.000115                4
13  checkoutservice     trace.to_currencyservice.duration_p95            25    0.000294    0.000186               25            2  2024-01-18T18:27:00+00:00            2  2024-01-18T18:40:00+00:00       0.000343       0.000153                4
14  checkoutservice        trace.to_currencyservice.row_count            25  121.400000   66.400000               25            2  2024-01-18T18:33:00+00:00            2  2024-01-18T18:36:00+00:00     131.000000       7.000000                4
15  checkoutservice       trace.to_emailservice.duration_mean            25    0.011901    0.000292               25            2  2024-01-18T18:38:00+00:00            2  2024-01-18T18:29:00+00:00       0.022364       0.000263                4
16  checkoutservice        trace.to_emailservice.duration_p95            25    0.085651    0.000425               25            2  2024-01-18T18:38:00+00:00            2  2024-01-18T18:29:00+00:00       0.087283       0.000396                4
17  checkoutservice           trace.to_emailservice.row_count            25   36.600000   19.400000               25            2  2024-01-18T18:33:00+00:00            2  2024-01-18T18:36:00+00:00      37.000000       3.000000                4
18  checkoutservice     trace.to_paymentservice.duration_mean            25    0.000518    0.000223               25            2  2024-01-18T18:41:00+00:00            2  2024-01-18T18:32:00+00:00       0.000873       0.000209                4
19  checkoutservice      trace.to_paymentservice.duration_p95            25    0.001689    0.000294               25            2  2024-01-18T18:43:00+00:00            2  2024-01-18T18:30:00+00:00       0.005423       0.000234                4```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Services of interest (reuse variable name)
comps = ["checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"]

# Ensure df_log and df_err exist from prior steps and timestamps are parsed to UTC
# Compute global statistics on full log.csv but restricted to the requested services
df_log_sub = df_log[df_log['cmdb_id'].isin(comps)].copy()

grp_full = df_log_sub.groupby(['cmdb_id','log_name'])['value']
global_count = grp_full.count().rename('global_count')
global_p95 = grp_full.quantile(0.95).rename('global_p95')
global_p5  = grp_full.quantile(0.05).rename('global_p5')

global_stats = pd.concat([global_count, global_p95, global_p5], axis=1).reset_index()

# Incident window (UTC)
start = pd.to_datetime("2024-01-18 18:21:04", utc=True)
end   = pd.to_datetime("2024-01-18 18:51:04", utc=True)

# Filter log rows to window (but anomalies compared to global thresholds computed above)
df_log_win = df_log_sub[(df_log_sub['timestamp'] >= start) & (df_log_sub['timestamp'] <= end)].copy()

# Merge global thresholds into windowed rows
df_log_win = df_log_win.merge(global_stats, on=['cmdb_id','log_name'], how='left')

# Flag anomalies relative to global p95/p5 (thresholds computed before window filtering)
df_log_win['above'] = (df_log_win['value'] > df_log_win['global_p95']).astype(int)
df_log_win['below'] = (df_log_win['value'] < df_log_win['global_p5']).astype(int)

# Window-level aggregates
agg_win = df_log_win.groupby(['cmdb_id','log_name']).agg(
    count_in_window = ('value','count'),
    count_above = ('above','sum'),
    count_below = ('below','sum'),
    max_in_window = ('value','max'),
    min_in_window = ('value','min')
).reset_index()

# Earliest timestamps for anomalies
earliest_above = df_log_win[df_log_win['above']==1].groupby(['cmdb_id','log_name'])['timestamp'].min().rename('earliest_above_ts')
earliest_below = df_log_win[df_log_win['below']==1].groupby(['cmdb_id','log_name'])['timestamp'].min().rename('earliest_below_ts')

# Combine global stats with window aggregates and earliest timestamps
result = global_stats.merge(agg_win, on=['cmdb_id','log_name'], how='left')
result = result.merge(earliest_above.reset_index(), on=['cmdb_id','log_name'], how='left')
result = result.merge(earliest_below.reset_index(), on=['cmdb_id','log_name'], how='left')

# Fill NaNs for counts (no samples in window -> set counts to 0)
result['count_in_window'] = result['count_in_window'].fillna(0).astype(int)
result['count_above'] = result['count_above'].fillna(0).astype(int)
result['count_below'] = result['count_below'].fillna(0).astype(int)

# Format earliest timestamps to ISO UTC strings or None
def fmt_iso(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).isoformat()

result['earliest_above'] = result['earliest_above_ts'].apply(fmt_iso)
result['earliest_below'] = result['earliest_below_ts'].apply(fmt_iso)
result.drop(columns=['earliest_above_ts','earliest_below_ts'], inplace=True)

# Compute totals and filter rows with anomalies
result['total_anomalies'] = result['count_above'] + result['count_below']
log_anomalies = result[result['total_anomalies']>0].copy()

# Select compact columns and sort/limit to top 20 by total_anomalies desc
log_anomalies = log_anomalies[[
    'cmdb_id','log_name','global_count','global_p95','global_p5',
    'count_in_window','count_above','earliest_above','count_below','earliest_below',
    'max_in_window','min_in_window','total_anomalies'
]].sort_values('total_anomalies', ascending=False).head(20).reset_index(drop=True)

# Ensure integer types for counts
log_anomalies['global_count'] = log_anomalies['global_count'].astype(int)
log_anomalies['count_in_window'] = log_anomalies['count_in_window'].astype(int)
log_anomalies['count_above'] = log_anomalies['count_above'].astype(int)
log_anomalies['count_below'] = log_anomalies['count_below'].astype(int)
log_anomalies['total_anomalies'] = log_anomalies['total_anomalies'].astype(int)

# Prepare top 20 error_logs rows for the same window (timestamp, cmdb_id, message), or empty if none
df_err_win = df_err[(df_err['timestamp'] >= start) & (df_err['timestamp'] <= end)].copy()
if df_err_win.empty:
    err_top_rows = pd.DataFrame(columns=['timestamp','cmdb_id','message'])
else:
    err_top_rows = df_err_win.sort_values('timestamp').head(20).copy()
    # Format timestamp to ISO UTC string
    err_top_rows['timestamp'] = err_top_rows['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    err_top_rows = err_top_rows[['timestamp','cmdb_id','message']].reset_index(drop=True)

# Final compact outputs (DataFrames)
log_anomalies, err_top_rows

```
Out[4]:
```
Summary (UTC):

Method: global P95/P5 were computed over the full log.csv for the requested services. Then log rows inside the incident window 2024-01-18 18:21:04 → 2024-01-18 18:51:04 UTC were compared to those global thresholds. An anomaly = value > global_p95 or value < global_p5. Only rows with anomalies are reported (top 20 limited).

Findings (only log.row_count showed anomalies for these services):

- checkoutservice · log.row_count
  - global_count=25, global_p95=109.8, global_p5=58.0
  - in-window samples=25
  - count_above_p95=2 (earliest above: 2024-01-18T18:33:00+00:00)
  - count_below_p5=2 (earliest below: 2024-01-18T18:36:00+00:00)
  - max_in_window=111, min_in_window=9
  - total_anomalies=4

- currencyservice · log.row_count
  - global_count=25, global_p95=2545.2, global_p5=2191.2
  - in-window samples=25
  - count_above_p95=2 (earliest above: 2024-01-18T18:30:00+00:00)
  - count_below_p5=2 (earliest below: 2024-01-18T18:24:00+00:00)
  - max_in_window=2608, min_in_window=160
  - total_anomalies=4

- emailservice · log.row_count
  - global_count=25, global_p95=36.6, global_p5=19.4
  - in-window samples=25
  - count_above_p95=2 (earliest above: 2024-01-18T18:33:00+00:00)
  - count_below_p5=2 (earliest below: 2024-01-18T18:36:00+00:00)
  - max_in_window=37, min_in_window=3
  - total_anomalies=4

- recommendationservice · log.row_count
  - global_count=25, global_p95=590.8, global_p5=514.6
  - in-window samples=25
  - count_above_p95=2 (earliest above: 2024-01-18T18:33:00+00:00)
  - count_below_p5=2 (earliest below: 2024-01-18T18:24:00+00:00)
  - max_in_window=609, min_in_window=45
  - total_anomalies=4

Error logs in the same window: empty (no error_logs rows found in the incident window).

Quick takeaway: Several services show oscillating log.row_count behavior during the window (typically two samples above global P95 and two below global P5). No textual error log entries were present for the same window.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id       log_name  global_count  global_p95  global_p5  count_in_window  count_above             earliest_above  count_below             earliest_below  max_in_window  min_in_window  total_anomalies
0        checkoutservice  log.row_count            25       109.8       58.0               25            2  2024-01-18T18:33:00+00:00            2  2024-01-18T18:36:00+00:00            111              9                4
1        currencyservice  log.row_count            25      2545.2     2191.2               25            2  2024-01-18T18:30:00+00:00            2  2024-01-18T18:24:00+00:00           2608            160                4
2           emailservice  log.row_count            25        36.6       19.4               25            2  2024-01-18T18:33:00+00:00            2  2024-01-18T18:36:00+00:00             37              3                4
3  recommendationservice  log.row_count            25       590.8      514.6               25            2  2024-01-18T18:33:00+00:00            2  2024-01-18T18:24:00+00:00            609             45                4, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```